In [1]:
# format data file (prompt-completion)
import json

# File paths
input_file_path = './data/raw_data.txt'
output_file_path = './data/kub_data_pc.jsonl'

# Read the input file
with open(input_file_path, 'r') as infile:
    lines = infile.readlines()

# Process the file to extract prompt-completion pairs
prompt_completion_pairs = []

for line in lines:
    if '```json' in line:
        # Extract prompt
        prompt = line.split('```json')[0].strip()
        # Extract completion
        completion = line.split('```json')[1].strip()
        if prompt and completion:
            pair = {
                "prompt": prompt,
                "completion": completion
            }
            prompt_completion_pairs.append(pair)

# Write the pairs to the output JSONL file
with open(output_file_path, 'w') as outfile:
    for pair in prompt_completion_pairs:
        json.dump(pair, outfile)
        outfile.write('\n')

pc_file = output_file_path

In [2]:
from openai import OpenAI
import warnings
import os

In [7]:
os.environ['OPENAI_API_KEY'] = ""
client = OpenAI()

In [14]:
# Upload the file
response_file = client.files.create(
    file=open(pc_file, "rb"),
    purpose="fine-tune"
)

In [15]:
# Create the fine-tune job
response_fine_tune = client.fine_tuning.jobs.create(
    training_file=response_file.id,
    model='babbage-002',
    suffix='fahim'
)
response_fine_tune

FineTuningJob(id='ftjob-8REpaQCljM1iqtyUm4sIVvyD', created_at=1723193704, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='babbage-002', object='fine_tuning.job', organization_id='org-waZhGhRYhuBa662pAubK6OhB', result_files=[], seed=1929008613, status='validating_files', trained_tokens=None, training_file='file-1eEErRZ5ZxaNFvZSdYrEXV2b', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix='fahim')

In [16]:
model_name = 'ft:babbage-002:personal:fahim:9uFm6jdr'

def ask_model_pc(question):
    completion = client.completions.create(
        model=model_name,
        prompt=question,
        max_tokens=150,
        temperature=0.7
    )
    return completion.choices[0].text.strip()

In [17]:
question = "Kubernetes Nodes with Network Unavailable"
ask_model_pc(question)

'{  "investigation": {    "steps": [      {        "description": "Check the status of the Kubernetes nodes",        "command": "kubectl get nodes"      },      {        "description": "Check the status of the Kubernetes nodes\' networks",        "command": "kubectl get networks -o wide"      },      {        "description": "Check the status of the Kubernetes pods running on the Kubernetes nodes",        "command": "kubectl get pods --all-namespaces -o wide"      },      {        "description": "Check the status of the Kubernetes services running on the Kubernetes nodes",        "command": "kubectl get svc --all-namespaces -o wide"      },      {        "description'